<a href="https://colab.research.google.com/github/leoncana/CreativeHub-DataEngineeringModule1/blob/main/homework/hw02_python_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 2 — Loops, functions and files

About 45-60 minutes. This time you work with a **real** Wavely export: all plays of 1 September 2026.

Columns: `event_id, event_ts, user_id, track_id, ms_played, device, country, skipped, shuffle`

In [2]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: /content/course/data


### Exercise 1 — Load the file
Read `DATA / "raw" / "plays_2026-09-01.csv"` with `csv.DictReader` into a list called `rows`.

In [3]:
import csv

file_path = DATA / "raw" / "plays_2026-09-01.csv"

with open(file_path, "r", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

print(len(rows), "rows")
print(rows[0])
print(rows[2])

4334 rows
{'event_id': '7a28c2bc7c9e', 'event_ts': '2026-09-01 00:00:09', 'user_id': 'U0525', 'track_id': 'T0028', 'ms_played': '226000', 'device': 'desktop', 'country': 'AT', 'skipped': 'false', 'shuffle': 'false'}
{'event_id': '02670d8cb977', 'event_ts': '2026-09-01 00:01:14', 'user_id': 'U0458', 'track_id': 'T0124', 'ms_played': '280000', 'device': 'mobile', 'country': 'CZ', 'skipped': 'false', 'shuffle': 'false'}


Notice: **every value is a string**, even `"226000"` and `"false"`. CSV files have no types — remember this, it matters below.

### Exercise 2 — Plays per device
Build `plays_per_device`, a dict device → number of rows.

In [4]:
# YOUR CODE HERE
plays_per_device = {}
for row in rows:
  device = row['device']
  plays_per_device[device] = plays_per_device.get(device, 0) + 1


print(plays_per_device)


{'desktop': 884, 'mobile': 2620, 'smart_speaker': 423, 'web': 407}


### Exercise 3 — Broken rows
A client bug sometimes sends `ms_played = -1`. Write `is_valid(row)` that returns `False` for those rows
(careful: the value is a string!). Count the invalid rows in `invalid_count`.

In [5]:
def is_valid(row):
  if row['ms_played'] == "-1":
    return False
  else:
    return True

invalid_count = 0
for row in rows:
  if is_valid(row) == False:
    invalid_count += 1


print(invalid_count)  # expected: 14

14


### Exercise 4 — Duplicates
Our app sometimes sends the same event twice. Count how often every `event_id` appears, then compute
`duplicate_ids`: how many event ids appear more than once.

In [6]:
# YOUR CODE HERE

duplicate_ids = 0

ids = {}
for row in rows:
  id = row['event_id']
  ids[id] = ids.get(id, 0) + 1
#print(ids)

for id in ids:
  if ids.get(id)>1:
    duplicate_ids +=1
print(duplicate_ids)


# expected: 64

64


### Exercise 5 — Listening hours per country
Build `hours_per_country`: country → total hours played, counting only rows that are valid **and** not skipped.
Round the values to 1 decimal at the end. (Hint: `row["skipped"] == "false"`.)

In [7]:
# YOUR CODE HERE

hours_per_country = {}

for row in rows:

    if not is_valid(row) or row["skipped"] != "false":
        continue

    country = row["country"]
    hours = int(row["ms_played"]) / 3_600_000

    hours_per_country[country] = hours_per_country.get(country, 0) + hours

# Round totals at the end
for country in hours_per_country:
    hours_per_country[country] = round(hours_per_country[country], 1)

print(hours_per_country)




{'AT': 45.0, 'CZ': 24.3, 'DE': 32.9, 'SK': 28.2, 'HU': 18.2, 'PL': 18.2}


### Exercise 6 — The busiest hour
`event_ts` looks like `"2026-09-01 18:22:05"`. The hour is characters 11 and 12: `row["event_ts"][11:13]`.
Find `busiest_hour` (as a string like `"19"`) and how many plays it had (`busiest_count`).

In [8]:
# YOUR CODE HERE

plays_per_hour = {}

for row in rows:
  hour = row['event_ts'][11:13]
  plays_per_hour[hour]= plays_per_hour.get(hour, 0) + 1

busiest_hour = max(plays_per_hour, key = plays_per_hour.get)
busiest_count = plays_per_hour[busiest_hour]
print(busiest_hour, busiest_count)

20 360


### Exercise 7 (Stretch) — One function for any day
Write `summarize_day(path)` that returns a dict with `file` (just the file name), `rows`, `unique_events`,
`skip_rate` (share of rows with skipped `"true"`, rounded to 3 decimals) and `top_device`.

Run it on the first three files of `sorted((DATA / "raw").glob("plays_*.csv"))`.

In [9]:
# YOUR CODE HERE
def summarize_day(path):
  with open(path, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

  skipped = 0
  devices = {}
  for row in rows:
    devices[row['device']] = devices.get(row['device'], 0) + 1
    if row['skipped'] == "true":
      skipped += 1
  return {
      "file": path.name,
      "rows": len(rows),
      "unique_events": len({row["event_id"] for row in rows}),
      "skip_rate": round(skipped / len(rows), 3),
      "top_device": max(devices, key = devices.get)
  }

for path in sorted((DATA / "raw").glob("plays_*.csv")):
  print(summarize_day(path))


{'file': 'plays_2026-09-01.csv', 'rows': 4334, 'unique_events': 4270, 'skip_rate': 0.308, 'top_device': 'mobile'}
{'file': 'plays_2026-09-02.csv', 'rows': 3902, 'unique_events': 3845, 'skip_rate': 0.282, 'top_device': 'mobile'}
{'file': 'plays_2026-09-03.csv', 'rows': 4338, 'unique_events': 4274, 'skip_rate': 0.297, 'top_device': 'mobile'}
{'file': 'plays_2026-09-04.csv', 'rows': 4039, 'unique_events': 3980, 'skip_rate': 0.291, 'top_device': 'mobile'}
{'file': 'plays_2026-09-05.csv', 'rows': 5262, 'unique_events': 5185, 'skip_rate': 0.293, 'top_device': 'mobile'}
{'file': 'plays_2026-09-06.csv', 'rows': 4958, 'unique_events': 4885, 'skip_rate': 0.293, 'top_device': 'mobile'}
{'file': 'plays_2026-09-07.csv', 'rows': 4091, 'unique_events': 4031, 'skip_rate': 0.29, 'top_device': 'mobile'}
{'file': 'plays_2026-09-08.csv', 'rows': 4463, 'unique_events': 4398, 'skip_rate': 0.302, 'top_device': 'mobile'}
{'file': 'plays_2026-09-09.csv', 'rows': 3956, 'unique_events': 3898, 'skip_rate': 0.285,